In [ ]:
# Var of interest covers cve, mammogram_past_2_yrs_40_ia_yes, skin_cancer_no, poor_mental_health_14_days_14_or_more_days_pct_diff, ep_nohsdp
library(tidyverse)
library(glmnet)
library(glmmTMB)
library(caret)
library(DHARMa)
library(readr)
library(dplyr)
library(janitor)
drop_redundant_yn_features <- function(df) {
  # Get all column names
  cols <- names(df)
  
  # Find Yes/No pairs by stripping the suffix
  yes_cols <- cols[grepl("_yes", cols)]
  no_cols  <- cols[grepl("_no",  cols)]
  
  # Get base names for each
  yes_bases <- sub("_diff$", "", sub("_yes", "", yes_cols))
  no_bases  <- sub("_diff$", "", sub("_no",  "", no_cols))
  
  # Find bases that have BOTH a Yes and No column
  paired_bases <- intersect(yes_bases, no_bases)
  
  cols_to_drop <- c()
  
  for (base in paired_bases) {
    yes_col <- yes_cols[yes_bases == base]
    no_col  <- no_cols[no_bases == base]
    
    # Count NAs in each
    yes_nas <- sum(is.na(df[[yes_col]]))
    no_nas  <- sum(is.na(df[[no_col]]))
    
    # Drop whichever has more NAs 
    # If tied drop "No" (keep "Yes")
    if (no_nas >= yes_nas) {
      cols_to_drop <- c(cols_to_drop, no_col)
    } else {
      cols_to_drop <- c(cols_to_drop, yes_col)
    }
  }
  
  cat("Dropping", length(cols_to_drop), "redundant columns:\n")
  #cat(paste(" ", cols_to_drop), sep = "\n")
  
  df[, !names(df) %in% cols_to_drop]
}

set.seed(100)

df <- read_csv("data/merged_with_svi.csv", show_col_types = FALSE) |> clean_names() 
#df <- read_csv("data/merged_with_svi.csv") |> clean_names()

# outbreak >=2 is with outbreak, vice versa.
# df$outbreak <- as.integer(df$outbreak >= def_outbreak)

# SVI specific, because it does not include the data for Mcculloch, Mclennan, Mcmullen and Dewitt, 
# if we continue the prev way of data processing it will remove all SVI related features, so we remove these 4 counties.
#df <- df[!df$county %in% c("Mcculloch", "Mclennan", "Mcmullen", "Dewitt"), ]

df <- df %>% filter(!county %in% c("Mcculloch", "Mclennan", "Mcmullen", "Dewitt", "Loving"))
df_raw <- df
df$density <- df$population*1.0/df$area_sqmi
df <- df %>% dplyr::select(-c("county", "area_sqmi"), -starts_with("m_"), -starts_with("mp_"), -starts_with("e_"), -starts_with("epl_"), -starts_with("spl_"), -starts_with("rpl_"), -starts_with("f_"))
df <- df[,colSums(is.na(df)) == 0]
df <- df %>% dplyr::select(-c("ep_minrty", "ep_hisp", "ep_afam", "ep_pov150", "ep_uninsur"))

df <- drop_redundant_yn_features(df)



In [ ]:
library(tidyverse)

selected <- c("cve", "mammogram_past_2_yrs_40_ia_yes", "skin_cancer_no", "poor_mental_health_14_days_14_or_more_days_pct_diff", "ep_nohsdp")

cve_cors <- tibble(
  variable = setdiff(names(df), selected),
  rho = sapply(setdiff(names(df), selected),
               function(v) cor(df$cve, df[[v]], method = "spearman", use = "pairwise.complete.obs"))
) %>%
  mutate(abs_rho = abs(rho)) %>%
  arrange(desc(abs_rho))

print(cve_cors, n = 25) 

mam_cors <- tibble(
  variable = setdiff(names(df), selected),
  rho = sapply(setdiff(names(df), selected),
               function(v) cor(df$mammogram_past_2_yrs_40_ia_yes, df[[v]], method = "spearman", use = "pairwise.complete.obs"))
) %>%
  mutate(abs_rho = abs(rho)) %>%
  arrange(desc(abs_rho))
print(mam_cors, n = 25)   # top 25 strongest correlates of mammogram_past_2_yrs_40_ia_yes

poor_cors <- tibble(
  variable = setdiff(names(df), selected),
  rho = sapply(setdiff(names(df), selected),
               function(v) cor(df$poor_mental_health_14_days_14_or_more_days_pct_diff, df[[v]], method = "spearman", use = "pairwise.complete.obs"))
) %>%
  mutate(abs_rho = abs(rho)) %>%
  arrange(desc(abs_rho))
print(poor_cors, n = 25)  

hsbp_cors <- tibble(
  variable = setdiff(names(df), selected),
  rho = sapply(setdiff(names(df), selected),
               function(v) cor(df$ep_nohsdp, df[[v]], method = "spearman", use = "pairwise.complete.obs"))
) %>%
  mutate(abs_rho = abs(rho)) %>%
  arrange(desc(abs_rho))
print(hsbp_cors, n = 25)  